# 02 — The Born Rule on Real Hardware

In `01/03–04` you sampled the Born rule on a simulator; now you read it straight off a real QPU and meet the device's read-out bias.

**Prerequisites:** `01/04_shot_noise_and_first_circuit`, `05/01_using_a_real_quantum_computer`.

**What you'll be able to do**
- Run a single-qubit state through `SamplerV2` and read its measurement counts back as probabilities.
- See read-out bias bend the device's split away from the ideal 50/50.
- Watch statistical scatter shrink as you add shots, while a systematic bias floor stays put.

You have already derived the Born rule and watched shot noise on the simulator, where the answer was exact. Here the same `|+>` state runs on a superconducting chip, and the counts come back with a fingerprint the simulator never showed: a small, *systematic* lean toward one outcome. Meeting that bias head-on is a milestone — it is the honest reason the course moved from pure states to density matrices, and the thing error mitigation later sets out to remove. The `USE_HARDWARE` switch keeps everything runnable on your laptop today and reaches a real device the moment you flip it.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qot_course import viz
from qot_course.colors import COLORS
from qiskit import QuantumCircuit
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2
from qot_course.hardware.runtime import select_backend
from qot_course.quantum.states import born_probabilities, KET_PLUS

np.random.seed(42)
viz.use_course_style()

In [ ]:
USE_HARDWARE = False  # flip to True after you've set up credentials (see 05/01)
backend, label, is_real = select_backend(use_hardware=USE_HARDWARE)
print(f"Running on: {label}  (real hardware = {is_real})")

## 1 · |+> on the device

We build `|+>` with a single Hadamard, run it through the `SamplerV2` primitive, and compare the device's counts to the ideal Born probabilities. The state `|+> = (|0> + |1>)/sqrt(2)` is the cleanest possible test: textbook quantum mechanics says a measurement in the computational basis lands on `0` and `1` with probability exactly one-half each. Any departure we see is the device speaking, not the mathematics.

In [ ]:
qc = QuantumCircuit(1, 1)
qc.h(0)                      # |0> -> |+>
qc.measure(0, 0)
isa = generate_preset_pass_manager(backend=backend, optimization_level=1).run(qc)
counts = SamplerV2(mode=backend).run([isa], shots=4096).result()[0].data.c.get_counts()

ideal = born_probabilities(KET_PLUS)           # {"0": 0.5, "1": 0.5}
total = sum(counts.values())
measured = {k: counts.get(k, 0) / total for k in ("0", "1")}
print("ideal   :", ideal)
print("measured:", {k: round(v, 3) for k, v in measured.items()})

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.5))
x = np.arange(2); w = 0.38
ax.bar(x - w / 2, [ideal["0"], ideal["1"]], width=w, label="ideal", color=COLORS["muted"])
ax.bar(x + w / 2, [measured["0"], measured["1"]], width=w, label="measured", color=COLORS["quantum"])
ax.axhline(0.5, color=COLORS["highlight"], linestyle="--", label="fair 50/50")
ax.set_xticks(x); ax.set_xticklabels(["0", "1"])
ax.set_ylabel("probability")
ax.set_title("Born rule for |+>: ideal vs a real device", pad=12)
ax.legend()
plt.show()

**Read the figure.** The device split is not exactly 50/50 — a read-out asymmetry biases `0` against `1`, so one bar sits a little above the dashed fair line and the other a little below. Offline, this is the *modeled* FakeManila read-out error baked into the noisy simulator; flip `USE_HARDWARE=True` and the very same gap becomes the literal device's bias. This systematic lean — present even with thousands of shots — is the honest reason `01/05` moved us from pure states to density matrices, which can describe exactly this kind of imperfect, mixed outcome.

### A real run on IBM hardware

On **`ibm_marrakesh`** (156-qubit Heron processor, 2026-05-30), measuring `|+>` returned **P(0) = 0.491, P(1) = 0.509** — a small but genuine read-out lean off the ideal 50/50. Set `USE_HARDWARE = True` to reproduce on the least-busy QPU your account can reach.

## 2 · Shot noise on hardware

Re-run the same `|+>` circuit at increasing shot counts. Two effects pull on the result at once. The *statistical* deviation from 0.5 — the scatter you met as shot noise in `01/04` — shrinks like $1/\sqrt{\mathrm{shots}}$ as you average over more measurements. But a *systematic* bias floor remains no matter how many shots you take: that floor is the device's read-out error, and no amount of sampling washes it out. Telling these two apart is the whole game of error characterization.

In [ ]:
shot_grid = [256, 1024, 8192]
deviations = []
for shots in shot_grid:
    c = SamplerV2(mode=backend).run([isa], shots=shots).result()[0].data.c.get_counts()
    tot = sum(c.values())
    deviations.append(abs(c.get("0", 0) / tot - 0.5))
    print(f"{shots:5d} shots: |P(0) - 0.5| = {deviations[-1]:.4f}")

fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.plot(shot_grid, deviations, "o-", color=COLORS["quantum"], label="|P(0) - 0.5|")
guide = deviations[0] * np.sqrt(shot_grid[0]) / np.sqrt(shot_grid)
ax.plot(shot_grid, guide, "--", color=COLORS["flow"], label=r"$1/\sqrt{\mathrm{shots}}$ guide")
ax.set_xscale("log"); ax.set_xlabel("shots"); ax.set_ylabel("deviation from 0.5")
ax.set_title("Statistical scatter shrinks; a device bias floor remains", pad=12)
ax.legend()
plt.show()

**Read the figure.** As shots grow, the statistical scatter falls and the measured deviation drops toward the $1/\sqrt{\mathrm{shots}}$ guide. What does *not* fall away is the device bias — the residual lean that sampling alone cannot remove. That residual is precisely the target of error mitigation. (Note: with only three points and real noise, the trend here is illustrative, not a precise fit — the guide line shows the shape you expect, not a fitted law.)

## Your turn

- Replace `qc.h(0)` with `qc.ry(theta, 0)` for a `theta` of your choice and compare the measured `P(0)` to the prediction `cos**2(theta/2)` you derived in `01/03`. How close does the device land, and does the gap grow as you tilt the state?
- Flip `USE_HARDWARE=True` (with credentials set per `05/01`) and re-run. The bias you see is now the real chip's, not a model's.
- Print `isa.layout` to see which physical qubit actually ran your circuit. Choose a different qubit on the device and check whether the read-out bias changes — read-out error is per-qubit, so it often does.

## What you built

- A real-hardware Born-rule run end to end: build `|+>`, transpile it to an ISA circuit, sample it with `SamplerV2`, and read the counts back as probabilities.
- A side-by-side picture of ideal vs measured that makes read-out bias visible as a departure from the fair 50/50 line.
- A shot sweep that separates the two error sources you now know by name: statistical scatter that averages away, and a systematic device bias floor that does not.

## What's next

Next you go beyond single probabilities and reconstruct a full quantum state on hardware — **tomography** — then score how close the reconstructed state is to the ideal with **fidelity**, turning the bias you met here into a quantitative measure.

## References

- M. A. Nielsen & I. L. Chuang, *Quantum Computation and Quantum Information*, ch. 2 — the Born rule and measurement (Cambridge University Press, 2010).
- Qiskit IBM Runtime documentation — `SamplerV2`, transpilation, and reading counts: <https://docs.quantum.ibm.com>.

**Previous:** `notebooks/05_RealQuantumHardware/01_using_a_real_quantum_computer.ipynb` · **Next:** `notebooks/05_RealQuantumHardware/03_tomography_and_fidelity_on_real_hardware.ipynb`